<a href="https://colab.research.google.com/github/yoginikumar0608-gh/GenZSpace/blob/main/AI_Interview_Panel_CrewAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U crewai litellm groq python-dotenv

In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("Groq API key loaded successfully ✅")

Enter your Groq API key: ··········
Groq API key loaded successfully ✅


In [3]:
from crewai import LLM

llm = LLM(
    model="openai/openai/gpt-oss-120b",
    custom_openai=True,
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.7
)

print("======================================")
print("CrewAI LLM READY ✅")
print("======================================")
print("Model:", llm.model)
print("Provider: Groq")

CrewAI LLM READY ✅
Model: openai/gpt-oss-120b
Provider: Groq


In [4]:
from crewai import Agent

technical_interviewer = Agent(
    role="Technical Interviewer",

    goal=(
        "Conduct a professional and interactive technical interview. "
        "Ask one question at a time, evaluate the candidate's previous "
        "answer, and adapt the next question according to the candidate's "
        "knowledge and performance."
    ),

    backstory="""
You are an experienced technical interviewer who interviews
engineering students and entry-level candidates.

You specialize in Python and general programming fundamentals.

You are friendly, professional, and conversational.

You ask exactly one question at a time.

You carefully evaluate each candidate answer.

If the candidate answers correctly, gradually increase the difficulty.

If the candidate struggles, ask an easier related question.

You never overwhelm the candidate with multiple questions.
""",

    llm=llm,

    verbose=False,

    max_retry_limit=1
)

print("Technical Interviewer created successfully ✅")

Technical Interviewer created successfully ✅


In [5]:
from crewai import Task, Crew, Process

# ============================================================
# INTERVIEW STATE
# ============================================================

conversation_history = []
question_number = 0


# ============================================================
# INTERVIEW FUNCTION
# ============================================================

async def interview_turn(candidate_answer=None):

    global conversation_history
    global question_number

    # --------------------------------------------------------
    # Save candidate answer
    # --------------------------------------------------------

    if candidate_answer is not None:
        conversation_history.append(
            f"Candidate: {candidate_answer}"
        )

    # --------------------------------------------------------
    # Prepare conversation history
    # --------------------------------------------------------

    if conversation_history:
        conversation_text = "\n".join(conversation_history)
    else:
        conversation_text = "No previous conversation. This is the beginning."

    # --------------------------------------------------------
    # Increase question number
    # --------------------------------------------------------

    question_number += 1

    # --------------------------------------------------------
    # CREATE TASK
    # --------------------------------------------------------

    interview_task = Task(

        description=f"""
You are conducting an interactive technical interview.

CURRENT QUESTION NUMBER:
{question_number}

CONVERSATION SO FAR:
{conversation_text}

YOUR RESPONSIBILITIES:

1. Read the candidate's latest answer carefully.
2. If a candidate answer exists, give very brief feedback.
3. Determine whether the candidate understood the topic.
4. Adapt the difficulty of the next question.
5. Ask exactly ONE technical question.
6. Never ask multiple questions.
7. Never give the complete answer to the previous question.
8. Keep the interview natural and conversational.
9. Focus mainly on Python and programming fundamentals.
10. Start with beginner-level questions and gradually increase difficulty.

IMPORTANT:

If this is the first question, do not provide feedback.

Simply begin the interview with one beginner-friendly Python question.

RESPONSE FORMAT:

Brief feedback: <very short feedback>

Next Question: <exactly one question>

Do NOT add another question.
""",

        expected_output=(
            "Brief feedback followed by exactly one "
            "technical interview question."
        ),

        agent=technical_interviewer
    )

    # --------------------------------------------------------
    # CREATE CREW
    # --------------------------------------------------------

    interview_crew = Crew(

        agents=[technical_interviewer],

        tasks=[interview_task],

        process=Process.sequential,

        verbose=False
    )

    # --------------------------------------------------------
    # RUN CREW ASYNCHRONOUSLY
    # --------------------------------------------------------

    result = await interview_crew.kickoff_async()

    # --------------------------------------------------------
    # Convert result to text
    # --------------------------------------------------------

    response = str(result.raw)

    # --------------------------------------------------------
    # Save AI response
    # --------------------------------------------------------

    conversation_history.append(
        f"AI Interviewer: {response}"
    )

    return response


print("Interactive interview engine created successfully ✅")

Interactive interview engine created successfully ✅


In [6]:
conversation_history = []
question_number = 0

print("=" * 65)
print("             🤖 AI TECHNICAL INTERVIEW")
print("=" * 65)

print()
print("AI Interviewer:")

first_response = await interview_turn()

print(first_response)

             🤖 AI TECHNICAL INTERVIEW

AI Interviewer:
Brief feedback: 

Next Question: What will be printed when you run the following Python code?

```python
message = "Hello, World!"
print(message[:5])
```


In [7]:
# ============================================================
# RESET INTERVIEW
# ============================================================

conversation_history = []
question_number = 0


# ============================================================
# INTERVIEW HEADER
# ============================================================

print()
print("=" * 70)
print("                 🤖 AI INTERVIEW PANEL")
print("=" * 70)

print()
print("Technical Interviewer")
print("-" * 70)

print()
print("Type 'quit' at any time to end the interview.")
print()


# ============================================================
# FIRST QUESTION
# ============================================================

response = await interview_turn()

print("AI:")
print(response)


# ============================================================
# INTERACTIVE LOOP
# ============================================================

while True:

    print()
    candidate_answer = input("You: ")

    # --------------------------------------------------------
    # EXIT
    # --------------------------------------------------------

    if candidate_answer.strip().lower() in [
        "quit",
        "exit",
        "stop"
    ]:

        print()
        print("=" * 70)
        print("AI: Thank you for participating in the interview.")
        print("AI: Your interview has been ended.")
        print("=" * 70)

        break

    # --------------------------------------------------------
    # GET NEXT AI RESPONSE
    # --------------------------------------------------------

    response = await interview_turn(candidate_answer)

    print()
    print("AI:")
    print(response)


                 🤖 AI INTERVIEW PANEL

Technical Interviewer
----------------------------------------------------------------------

Type 'quit' at any time to end the interview.

AI:
Brief feedback: 

Next Question: How would you create a list containing the numbers from 1 to 10 in Python?

You: hi

AI:
Brief feedback: Thanks for the greeting! Let's start with a basic list question.

Next Question: How do you create an empty list in Python?

You: a[]

AI:
Brief feedback: The syntax `a[]` isn’t valid for creating an empty list; the correct literal is `[]`.

Next Question: How would you create a list that contains just the number 5 in Python?

You: a[5]

AI:
Brief feedback: Close, but list literals don’t include a variable name before the brackets.

Next Question: What is the correct syntax to create a list that contains only the number 5?

You: can i get answer for previous question

AI:
Brief feedback: I understand you’d like the answer, but let’s give it a try yourself – you’ve got 